In [1]:
pip install PyMuPDF pdfminer.six pandas

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ------------ --------------------------- 2.1/6.6 MB 14.6 MB/s eta 0:00:01
   ------------------------------------ --- 6.0/6.6 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 15.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [16]:
import fitz  # PyMuPDF
import os
import pandas as pd
import re

In [3]:
def extract_and_clean_contract(pdf_path):
    """
    Extracts text from a SaaS contract PDF using PyMuPDF
    and applies legal-NLP friendly customizations.
    
    Input:
        pdf_path (str): path to PDF file
    Output:
        cleaned_text (str): cleaned, structured contract text
    """
    
    doc = fitz.open(pdf_path)
    full_text = ""

    for page_num, page in enumerate(doc):
        # --- Page marker for traceability & explainability ---
        full_text += f"\n\n--- Page {page_num + 1} ---\n\n"
        
        page_text = page.get_text("text")
        full_text += page_text

    # -------------------------------
    # Cleaning & normalization
    # -------------------------------

    # Normalize excessive newlines
    full_text = re.sub(r'\n{3,}', '\n\n', full_text)

    # Normalize spaces and tabs
    full_text = re.sub(r'[ \t]+', ' ', full_text)

    # Fix broken quotation definitions (common in contracts)
    full_text = re.sub(r'“\s*\n\s*', '“', full_text)
    full_text = re.sub(r'\s*\n\s*”', '”', full_text)

    # Remove repeated headers/footers (basic heuristic)
    full_text = re.sub(r'Software as a Service Agreement.*Page \d+ of \d+', '', full_text)

    return full_text.strip()


In [20]:
pdf_path = "E:\All Contract Text File\Contract_01.txt"

contract_text = extract_and_clean_contract(pdf_path)

print(contract_text[:200])  # preview first 2000 characters


--- Page 1 ---

--- Page 1 ---
Software as a Service Agreement (US December 2022) 
Page 1 of 8 
 
 
Software as a Service Agreement 
 
This Software as a Service Agreement (the
“Agreement”) is between


In [25]:
input_folder = "E:\All Saas Contract Updated"
output_folder = "E:\All Contract Text File - Copy"


In [26]:
import fitz
import re
import os

def extract_and_clean_contract(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""

    for page_num, page in enumerate(doc):
        text += f"\n\n--- Page {page_num + 1} ---\n\n"
        text += page.get_text("text")

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


In [27]:
all_contracts_text = {}

for filename in os.listdir(input_folder):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(input_folder, filename)
        cleaned_text = extract_and_clean_contract(pdf_path)
        all_contracts_text[filename] = cleaned_text


In [28]:
os.makedirs(output_folder, exist_ok=True)

for filename, text in all_contracts_text.items():
    txt_name = filename.replace(".pdf", ".txt")
    txt_path = os.path.join(output_folder, txt_name)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)


In [29]:
len(all_contracts_text)  # should be 25


25

In [30]:
CLAUSE_PATTERNS = [
    r'^\d+\.\s+[A-Z].+',              # 1. Definitions
    r'^\d+\.\d+\s+[A-Z].+',            # 1.1 Sub clauses (optional)
    r'^[A-Z][A-Z\s]{5,}$',             # ALL CAPS headings
    r'^Exhibit\s+[A-Z0-9]+',            # Exhibit A
    r'^Schedule\s+[A-Z0-9]+'            # Schedule 1
]


In [31]:
import re
import pandas as pd
from typing import List, Dict, Tuple

In [33]:
# STEP 1: EXAMINE YOUR CONTRACT STRUCTURE (Run this first!)
# ============================================================================

def peek_at_contract(contract_text: str, lines: int = 50):
    """
    Look at the beginning of a contract to understand its structure.
    Run this on 2-3 contracts to identify the pattern.
    """
    print("=" * 80)
    print("FIRST 50 LINES OF CONTRACT:")
    print("=" * 80)
    lines_list = contract_text.split('\n')[:lines]
    for i, line in enumerate(lines_list):
        print(f"{i:3d}: {line}")
    print("=" * 80)

# Run this to see what you're working with:
# peek_at_contract(all_contracts_text[0])


In [34]:
# STEP 2: CLEAN THE TEXT (Remove PDF artifacts)
# ============================================================================

def clean_contract_text(text: str) -> str:
    """
    Removes page numbers, headers, footers, and weird spacing from PDF extraction.
    """
    # Remove page numbers (standalone numbers on lines)
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    
    # Remove "Page X of Y" patterns
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    
    # Remove excessive whitespace
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    
    # Remove weird PDF characters
    text = text.replace('\x0c', '')  # Form feed character
    
    # Standardize line endings
    text = text.replace('\r\n', '\n')
    
    return text.strip()

# Clean all contracts
cleaned_contracts = [clean_contract_text(contract) for contract in all_contracts_text]
print(f"✅ Cleaned {len(cleaned_contracts)} contracts")


✅ Cleaned 25 contracts


In [36]:
def split_into_lines(contract_text):
    lines = contract_text.split("\n")
    return [line.strip() for line in lines if line.strip()]


In [37]:
annotated_contract = [
    ("1. Confidentiality", "B-CLAUSE"),
    ("The parties agree to keep information confidential.", "I-CLAUSE"),
    ("This obligation shall survive termination.", "I-CLAUSE"),
    ("2. Termination", "B-CLAUSE"),
    ("Either party may terminate the agreement.", "I-CLAUSE"),
]


In [38]:
import re

def extract_features(lines, i):
    line = lines[i]
    
    features = {
        "bias": 1.0,
        "is_upper": line.isupper(),
        "starts_with_digit": bool(re.match(r'^\d+', line)),
        "has_decimal": bool(re.match(r'^\d+\.\d+', line)),
        "line_length": len(line),
        "contains_shall": "shall" in line.lower(),
    }

    # Context (previous line)
    if i > 0:
        features["prev_starts_digit"] = bool(re.match(r'^\d+', lines[i-1]))
    else:
        features["BOS"] = True  # Beginning of document
    
    return features


In [39]:
def prepare_dataset(annotated_data):
    lines = [text for text, label in annotated_data]
    labels = [label for text, label in annotated_data]
    
    X = [extract_features(lines, i) for i in range(len(lines))]
    y = labels
    
    return [X], [y]  # CRF expects list of sequences


In [40]:
!pip install sklearn-crfsuite



   -------------------- ------------------- 1/2 [sklearn-crfsuite]
   ---------------------------------------- 2/2 [sklearn-crfsuite]



In [41]:
import sklearn_crfsuite

X_train, y_train = prepare_dataset(annotated_contract)

crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    max_iterations=100
)

crf.fit(X_train, y_train)


CRF(algorithm='lbfgs', max_iterations=100)

In [1]:
#Vectorisation

In [2]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---- ----------------------------------- 2.6/24.4 MB 16.9 MB/s eta 0:00:02
   ------------ --------------------------- 7.6/24.4 MB 21.1 MB/s eta 0:00:01
   ---------------------- ----------------- 13.9/24.4 MB 24.9 MB/s eta 0:00:01
   --------------------------------- ------ 20.4/24.4 MB 26.7 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 25.3 MB/s eta 0:00:00


In [3]:
import gensim
import os

In [11]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Anwar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [16]:
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Anwar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Anwar\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [22]:
#⚠ Important (Very Important for You)

# What you did is useful for:

# ✅ Word2Vec
# ✅ Topic modeling (LDA)
# ✅ Language modeling
# ✅ Word embeddings
# ✅ Text classification

# BUT ❌ NOT ideal for clause segmentation.

# Because:

# Clause segmentation is structural (line-based)

# You used sentence-based tokenization

import os
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

all_contracts_text = []

folder_path = r'E:\All Contract Text File - Copy'

for filename in os.listdir(folder_path):
    
    file_path = os.path.join(folder_path, filename)
    
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        corpus = f.read()
    
    raw_sent = sent_tokenize(corpus)
    
    for sent in raw_sent:
        all_contracts_text.append(simple_preprocess(sent))


In [23]:
#all_contracts_text

In [27]:
!pip install transformers torch scikit-learn pandas


In [28]:
import pandas as pd

data = {
    "clause_text": [
        "Seller shall indemnify the Buyer against all losses.",
        "Payment shall be made within 30 days.",
        "This Agreement shall be governed by Indian law."
    ],
    "risk_label": [
        "High",
        "Medium",
        "Low"
    ]
}

df = pd.DataFrame(data)
df.head()


,clause_text,risk_label
0,Seller shall indemnify the Buyer against all l...,High
1,Payment shall be made within 30 days.,Medium
2,This Agreement shall be governed by Indian law.,Low


In [29]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["risk_label"])

df.head()


,clause_text,risk_label,label
0,Seller shall indemnify the Buyer against all l...,High,0
1,Payment shall be made within 30 days.,Medium,2
2,This Agreement shall be governed by Indian law.,Low,1


In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(df["label"].unique())
)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\Anwar\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Anwar\.cache\huggingface\hub\models--nlpaueb--legal-bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [31]:
encodings = tokenizer(
    df["clause_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=256
)


In [32]:
import torch

class ContractDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset = ContractDataset(encodings, df["label"].tolist())


In [33]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


In [38]:
from transformers import TrainingArguments, Trainer